Cài đặt dependencies (chạy trong terminal hoặc notebook):
    pip install vllm transformers torch accelerate -U

Hoặc trong Jupyter:
    !pip install vllm transformers torch accelerate -U

In [ ]:
!nvidia-smi

In [ ]:
!pip install vllm transformers accelerate --no-cache-dir

In [ ]:
import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import time
from typing import List, Dict, Optional
import json

print("="*70)
print("SYSTEM INFORMATION")
print("="*70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("="*70)

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"

# vLLM configuration
CONFIG = {
    "model": MODEL_NAME,
    "tensor_parallel_size": 1,
    "dtype": "half",
    "max_model_len": 4096,
    "gpu_memory_utilization": 0.9,
    "trust_remote_code": True,
}

print("\nConfiguration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
import os
import subprocess

# Tìm vị trí thực tế của libcuda.so.1
result = subprocess.run(['find', '/', '-name', 'libcuda.so.1'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
cuda_libs = result.stdout.strip().split('\n')

if cuda_libs:
    target_lib = cuda_libs[0] # Lấy đường dẫn đầu tiên tìm thấy (thường là /usr/lib/x86_64-linux-gnu/libcuda.so.1)
    lib_dir = os.path.dirname(target_lib)
    
    # Tạo symlink libcuda.so -> libcuda.so.1
    link_name = os.path.join(lib_dir, 'libcuda.so')
    if not os.path.exists(link_name):
        print(f"Creating symlink: {link_name} -> {target_lib}")
        os.system(f"ln -s {target_lib} {link_name}")
    else:
        print(f"Symlink found at: {link_name}")
        
    # Thêm đường dẫn vào biến môi trường để trình biên dịch tìm thấy
    os.environ['LIBRARY_PATH'] = f"{lib_dir}:{os.environ.get('LIBRARY_PATH', '')}"
    os.environ['LD_LIBRARY_PATH'] = f"{lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
else:
    print("WARNING: Could not find libcuda.so.1 on this system.")

In [ ]:
print("\nLoading model...")
start = time.time()

llm = LLM(**CONFIG)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print(f"Loaded in {time.time()-start:.1f}s")

In [ ]:
def format_chat_prompt(messages: List[Dict[str, str]]) -> str:
    """Format messages theo Qwen chat template"""
    return tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

def generate_response(
    prompt: str,
    temperature: float = 0.7,
    max_tokens: int = 512,
    top_p: float = 0.9,
) -> str:
    """Generate response từ prompt"""
    
    sampling_params = SamplingParams(
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p,
        repetition_penalty=1.1,
    )
    
    outputs = llm.generate([prompt], sampling_params)
    return outputs[0].outputs[0].text

def chat(user_message: str, system_prompt: Optional[str] = None) -> str:
    """Chat interface đơn giản"""
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})
    
    prompt = format_chat_prompt(messages)
    return generate_response(prompt)

In [ ]:
print("\n" + "="*70)
print("Simple Chat")
print("="*70)

question = "Giải thích vLLM là gì và tại sao nó nhanh?"
print(f"\nQuestion: {question}")

response = chat(question)
print(f"\nResponse:\n{response}")

In [ ]:
print("\n" + "="*70)
print("Batch Inference")
print("="*70)

questions = [
    "Viết một hàm Python tính số Fibonacci",
    "Giải thích PagedAttention trong vLLM",
    "Ưu điểm của continuous batching là gì?",
]

# Format tất cả prompts
prompts = [
    format_chat_prompt([{"role": "user", "content": q}]) 
    for q in questions
]

# Batch inference
sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=256,
    top_p=0.9,
)

print(f"\nProcessing {len(questions)} questions in batch...")
start = time.time()

outputs = llm.generate(prompts, sampling_params)

elapsed = time.time() - start
print(f"Completed in {elapsed:.2f}s ({elapsed/len(questions):.2f}s per question)")

for i, output in enumerate(outputs):
    print(f"\n{'='*70}")
    print(f"Question {i+1}: {questions[i]}")
    print(f"{'-'*70}")
    print(output.outputs[0].text)

In [ ]:
def benchmark_vllm_streaming(llm_instance, prompt: str, num_requests: int = 100):
    print(f"Bat dau test tai voi {num_requests} requests (Streaming Mode)...")
    print(f"Prompt: {prompt}\n")

    sampling_params = SamplingParams(
        temperature=0.7, 
        max_tokens=512,
        top_p=0.9
    )

    ttft_list = []
    tps_list = []
    
    
    for i in range(num_requests):
        print(f"Request {i+1}/{num_requests} processing...", end="\r")
        
        start_time = time.perf_counter()
        
        
        requests_outputs = llm_instance.generate([prompt], sampling_params, use_tqdm=False)
        output = requests_outputs[0]
        
        end_time = time.perf_counter()
        
        # Chỉ số
        generated_text = output.outputs[0].text
        num_tokens = len(output.outputs[0].token_ids)
        duration = end_time - start_time
        
        # TTFT: Trong blocking mode, không đo được thời điểm token đầu tiên.
        # Chúng ta đành chấp nhận chỉ đo Speed.
        speed = num_tokens / duration if duration > 0 else 0
        
        # Lưu lại
        tps_list.append(speed)
        
        # Format in ra (TTFT đành để N/A vì giới hạn API đồng bộ)
        print(f"Request {i+1}/{num_requests} | Tokens: {num_tokens} | TTFT: N/A (Batch Mode) | Speed: {speed:.2f} tok/s")

    # Tổng kết
    avg_speed = np.mean(tps_list)
    print("\n" + "="*40)
    print("KET QUA BENCHMARK (Sequential)")
    print("="*40)
    print(f"Toc do trung binh: {avg_speed:.2f} tokens/s")
    print("luu y: TTFT khong do duoc voi vllm.LLM() class (blocking).")
    print("De do TTFT, can dung AsyncLLMEngine hoac OpenAI API Serve.")

In [ ]:
benchmark_results = benchmark_vllm_streaming(llm,"Tell me about AI in today economic", 100)

In [ ]:
def benchmark_vllm_with_metrics(llm_instance, prompt: str, num_requests: int = 100):
    print(f"Bat dau test tai voi {num_requests} requests...")
    print(f"Prompt: {prompt}\n")

    # Cấu hình không ignore EOS để độ dài token thay đổi tự nhiên,
    # hoặc set max_tokens fix cứng nếu muốn stress test.
    sampling_params = SamplingParams(
        temperature=0.7, 
        max_tokens=512,
        top_p=0.9,
    )

    tps_list = []
    
    
    for i in range(num_requests):
        print(f"Request {i+1}/{num_requests}...", end="\r")
        
        start_time = time.perf_counter()
        
        # Gọi sinh text
        request_outputs = llm_instance.generate([prompt], sampling_params, use_tqdm=False)
        output = request_outputs[0]
        
        end_time = time.perf_counter()
        duration = end_time - start_time
        
        # Lấy thông tin metrics (nếu vLLM version mới có hỗ trợ)
        # output.metrics.first_token_time là cái chúng ta cần.
        # Nếu không có, ta sẽ fallback về N/A hoặc estimation.
        
        ttft_ms = "N/A"
        try:
            # Cố gắng lấy metrics từ vLLM output (chỉ có trên phiên bản mới)
            if hasattr(output, 'metrics') and output.metrics is not None:
                # time_in_queue + time_prefill...
                # Một số version vLLM expose metrics.first_token_time
                if hasattr(output.metrics, 'first_token_time'):
                    # first_token_time often absolute timestamp.
                    # arrival_time also absolute.
                    ftt = output.metrics.first_token_time
                    arrival = output.metrics.arrival_time
                    ttft_val = ftt - arrival
                    ttft_ms = f"{ttft_val * 1000:.2f}ms"
                else:
                    pass
        except Exception:
            pass

        # Tính toán Speed
        num_tokens = len(output.outputs[0].token_ids)
        speed = num_tokens / duration if duration > 0 else 0
        tps_list.append(speed)
        
        print(f"Request {i+1}/{num_requests} | Tokens: {num_tokens} | TTFT: {ttft_ms} | Speed: {speed:.2f} tok/s")

In [ ]:
prompt_test = "Tell me about AI in today economic"
benchmark_vllm_with_metrics(llm, prompt_test, num_requests=100)


# Chuyen sang call bang openai

In [ ]:
!pip install openai --no-cache-dir

In [ ]:
import sys
import subprocess
import time
import socket
import os
import threading
from openai import OpenAI
import numpy as np

In [ ]:
# PHẦN 1: TỰ ĐỘNG FIX LỖI MÔI TRƯỜNG (LIBCUDA)
def setup_environment():
    print(">>> Đang cấu hình môi trường CUDA...")
    env = os.environ.copy()
    
    # Tìm libcuda.so.1
    try:
        # Cách tìm nhanh hơn find /
        paths_to_check = [
            "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
            "/usr/lib64/libcuda.so.1",
            "/usr/local/cuda/lib64/stubs/libcuda.so"
        ]
        
        target_lib = None
        for p in paths_to_check:
            if os.path.exists(p):
                target_lib = p
                break
        
        # Nếu không thấy, thử tìm kỹ
        if not target_lib:
            res = subprocess.run(['find', '/usr', '-name', 'libcuda.so.1'], stdout=subprocess.PIPE, text=True)
            if res.stdout:
                target_lib = res.stdout.strip().split('\n')[0]

        if target_lib:
            lib_dir = os.path.dirname(target_lib)
            # Tạo symlink nếu cần (yêu cầu quyền write, thường ok trên Kaggle/Colab)
            link_path = os.path.join(lib_dir, 'libcuda.so')
            if not os.path.exists(link_path):
                try:
                    os.symlink(target_lib, link_path)
                    print(f"    Đã tạo symlink: {link_path}")
                except Exception:
                    pass
            
            # Cập nhật biến môi trường cho tiến trình con
            env['LD_LIBRARY_PATH'] = f"{lib_dir}:{env.get('LD_LIBRARY_PATH', '')}"
            print(f"    Đã cập nhật LD_LIBRARY_PATH: {lib_dir}")
        else:
            print("    CẢNH BÁO: Không tìm thấy libcuda.so.1")
            
    except Exception as e:
        print(f"    Lỗi setup môi trường: {e}")
        
    return env

In [ ]:
# PHẦN 2: QUẢN LÝ SERVER VLLM
def start_vllm_server(model_name, port=8000, env=None):
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_name,
        "--dtype", "half",               # T4 dùng FP16
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.5", # Cực kỳ an toàn cho T4
        "--max-num-seqs", "16",          # Giới hạn concurrency
        "--enforce-eager",               # Tắt CUDA Graph để tránh crash
        "--host", "0.0.0.0",
        "--port", str(port),
        "--trust-remote-code",
        "--disable-log-stats"
    ]
    
    print(f"\n>>> Đang khởi chạy Server (Port {port})...")
    # print("Command:", " ".join(cmd))
    
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        env=env,       # Truyền môi trường đã fix
        bufsize=1      # Line buffered
    )
    return process

def wait_and_monitor(process, port, timeout=300):
    start_time = time.time()
    print(">>> Đang chờ Server sẵn sàng...")
    
    while time.time() - start_time < timeout:
        # 1. Kiểm tra xem process còn sống không
        ret_code = process.poll()
        if ret_code is not None:
            print(f"\n!!! LỖI NGHIÊM TRỌNG: Server đã tắt đột ngột (Exit code: {ret_code})")
            print("-" * 20 + " LOG LỖI (STDERR) " + "-" * 20)
            # Đọc toàn bộ lỗi còn lại
            print(process.stderr.read())
            print("-" * 50)
            return False

        # 2. Kiểm tra kết nối port
        try:
            with socket.create_connection(("localhost", port), timeout=1):
                print(f"\n>>> Server đã ONLINE tại port {port}!")
                return True
        except (socket.timeout, ConnectionRefusedError):
            pass
            
        # 3. In log realtime nhẹ nhàng để user biết không bị treo
        line = process.stderr.readline()
        if line:
            if "Uvicorn running" in line or "Application startup" in line:
                print(f"[Server info] {line.strip()}")
            # Uncomment dòng dưới nếu muốn xem mọi log (sẽ khá spam)
            # print(f"[Log] {line.strip()}")
            
        time.sleep(0.5)
        
    print("\n!!! Time out: Server không phản hồi sau thời gian chờ.")
    return False

In [ ]:
# PHẦN 3: BENCHMARK CLIENT
def run_benchmark(port, model_name, num_requests=20):
    print(f"\n>>> Bắt đầu Benchmark Online ({num_requests} requests)...")
    
    client = OpenAI(api_key="EMPTY", base_url=f"http://localhost:{port}/v1")
    prompt = "Explain the concept of Artificial Intelligence in a concise paragraph."
    
    # Warmup
    try:
        client.chat.completions.create(
            model=model_name, messages=[{"role": "user", "content": "Hi"}], max_tokens=5
        )
    except Exception as e:
        print(f"Lỗi Warmup: {e}")
        return

    metrics = []
    
    for i in range(num_requests):
        start = time.perf_counter()
        first_token_time = None
        count = 0
        
        try:
            stream = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=256,
                temperature=0.7,
                stream=True
            )
            
            for chunk in stream:
                if chunk.choices[0].delta.content:
                    if first_token_time is None:
                        first_token_time = time.perf_counter()
                    count += 1
            
            end = time.perf_counter()
            
            if first_token_time:
                ttft = (first_token_time - start) * 1000
                total_time = end - first_token_time
                speed = count / total_time if total_time > 0 else 0
                
                print(f"Request {i+1}/{num_requests} | Tokens: {count:<4} | TTFT: {ttft:.2f}ms | Speed: {speed:.2f} tok/s")
                metrics.append((ttft, speed))
            else:
                print(f"Request {i+1} Failed (No output)")
                
        except Exception as e:
            print(f"Request {i+1} Error: {e}")

    if metrics:
        avg_ttft = np.mean([m[0] for m in metrics])
        avg_speed = np.mean([m[1] for m in metrics])
        print("\n" + "="*50)
        print(f"KẾT QUẢ CUỐI CÙNG ({model_name})")
        print("="*50)
        print(f"Avg TTFT:  {avg_ttft:.2f} ms")
        print(f"Avg Speed: {avg_speed:.2f} tokens/s")
        print("="*50)

In [ ]:
MODEL = "Qwen/Qwen3-0.6B"
PORT = 8000

# 1. Setup Env
custom_env = setup_environment()

# 2. Start Server
server_proc = start_vllm_server(MODEL, PORT, custom_env)

try:
    # 3. Wait & Benchmark
    if wait_and_monitor(server_proc, PORT):
        run_benchmark(PORT, MODEL, num_requests=100)
finally:
    # 4. Cleanup
    print("\n>>> Đang dọn dẹp (Tắt server)...")
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except:
        server_proc.kill()
    print(">>> Hoàn tất.")


# Thay the bang model khac 

In [ ]:
MODEL_NAME = "ibm-granite/granite-4.0-h-1b"

# vLLM configuration
CONFIG = {
    "model": MODEL_NAME,
    "tensor_parallel_size": 1,
    "dtype": "half",
    "max_model_len": 4096,
    "gpu_memory_utilization": 0.7,
    "trust_remote_code": True,
}

print("\nConfiguration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
import os
import subprocess

# Tìm vị trí thực tế của libcuda.so.1
result = subprocess.run(['find', '/', '-name', 'libcuda.so.1'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
cuda_libs = result.stdout.strip().split('\n')

if cuda_libs:
    target_lib = cuda_libs[0] # Lấy đường dẫn đầu tiên tìm thấy (thường là /usr/lib/x86_64-linux-gnu/libcuda.so.1)
    lib_dir = os.path.dirname(target_lib)
    
    # Tạo symlink libcuda.so -> libcuda.so.1
    link_name = os.path.join(lib_dir, 'libcuda.so')
    if not os.path.exists(link_name):
        print(f"Creating symlink: {link_name} -> {target_lib}")
        os.system(f"ln -s {target_lib} {link_name}")
    else:
        print(f"Symlink found at: {link_name}")
        
    # Thêm đường dẫn vào biến môi trường để trình biên dịch tìm thấy
    os.environ['LIBRARY_PATH'] = f"{lib_dir}:{os.environ.get('LIBRARY_PATH', '')}"
    os.environ['LD_LIBRARY_PATH'] = f"{lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
else:
    print("WARNING: Could not find libcuda.so.1 on this system.")

In [ ]:
print("\nLoading model...")
start = time.time()

llm = LLM(**CONFIG)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print(f"Loaded in {time.time()-start:.1f}s")

In [ ]:
def format_chat_prompt(messages: List[Dict[str, str]]) -> str:
    """Format messages theo Qwen chat template"""
    return tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

def generate_response(
    prompt: str,
    temperature: float = 0.7,
    max_tokens: int = 512,
    top_p: float = 0.9,
) -> str:
    """Generate response từ prompt"""
    
    sampling_params = SamplingParams(
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p,
        repetition_penalty=1.1,
    )
    
    outputs = llm.generate([prompt], sampling_params)
    return outputs[0].outputs[0].text

def chat(user_message: str, system_prompt: Optional[str] = None) -> str:
    """Chat interface đơn giản"""
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})
    
    prompt = format_chat_prompt(messages)
    return generate_response(prompt)

In [ ]:
print("\n" + "="*70)
print("Simple Chat")
print("="*70)

question = "Giải thích vllm là gì và tại sao nó nhanh?"
print(f"\nQuestion: {question}")

response = chat(question)
print(f"\nResponse:\n{response}")